In [0]:
# Databricks Notebook: ACID Proof without DBFS Root Access
from pyspark.sql.functions import col, lit, when, raise_error, expr, rand
# --- SETUP ---
# Add the '00_landing' schema in front of the table names
TABLE_NAME = "training_medallion.00_landing.risk_optimization_demo"
CLUSTER_TABLE = "training_medallion.00_landing.risk_liquid_demo"

print(f"--- PREPARING OPTIMIZATION PROOF FOR: {TABLE_NAME} ---")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
spark.sql(f"DROP TABLE IF EXISTS {CLUSTER_TABLE}")

# --- 1. PARTITIONING & SMALL FILES SIMULATION ---
print("\n[1] PARTITIONING: Logically dividing large datasets into folder hierarchies.")
spark.sql(f"""
    CREATE TABLE {TABLE_NAME} (
        incident_id INT,
        risk_type STRING,
        severity INT,
        year INT
    ) USING DELTA PARTITIONED BY (year)
""")

# Simulate the creation of many small files (data fragmentation)
for i in range(5):
    df = spark.range(100).withColumn("incident_id", (col("id") + i*100).cast("int")) \
                         .withColumn("risk_type", expr("CASE WHEN id % 2 = 0 THEN 'Fraud' ELSE 'Cyber' END")) \
                         .withColumn("severity", (rand() * 10).cast("int")) \
                         .withColumn("year", lit(2024)) \
                         .drop("id") # <-- FIX: Drop the auto-generated 'id' column
                         
    df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)

detail_before = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").select("numFiles").first()[0]
print(f"-> Evidence: Created {detail_before} fragmented small files inside the 'year=2024' partition.")

# --- 2. OPTIMIZE (FILE COMPACTION) & Z-ORDER ---
print("\n[2] OPTIMIZE & Z-ORDER: Merging tiny files and co-locating related data.")
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (risk_type)")

detail_after = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").select("numFiles").first()[0]
print(f"-> Evidence: Metadata overhead reduced. Files compacted from {detail_before} down to {detail_after} optimal-sized Parquet file(s).")

# --- 3. LIQUID CLUSTERING (DYNAMIC LAYOUT) ---
print("\n[3] LIQUID CLUSTERING: Dynamic layout that replaces Partitioning & Z-Order.")
spark.sql(f"""
    CREATE TABLE {CLUSTER_TABLE} (
        incident_id INT,
        risk_type STRING,
        date DATE
    ) USING DELTA CLUSTER BY (risk_type, date)
""")
print(f"-> Evidence: Table '{CLUSTER_TABLE}' created with Adaptable Layout.")

# --- 4. VACUUM (STORAGE CLEANUP) ---
print("\n[4] VACUUM: Safe storage cleanup with 7-day retention.")
versions = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}").count()
print(f"-> Evidence: Transaction log currently holds {versions} versions.")

# Run Vacuum with safe defaults
spark.sql(f"VACUUM {TABLE_NAME}")
print("-> Evidence: Vacuum executed successfully. Delta Lake strictly protects data younger than 7 days (168 hours) to guarantee Time Travel safety. Obsolete files older than 7 days will be physically deleted.")

--- PREPARING OPTIMIZATION PROOF FOR: training_medallion.00_landing.risk_optimization_demo ---

[1] PARTITIONING: Logically dividing large datasets into folder hierarchies.
-> Evidence: Created 5 fragmented small files inside the 'year=2024' partition.

[2] OPTIMIZE & Z-ORDER: Merging tiny files and co-locating related data.
-> Evidence: Metadata overhead reduced. Files compacted from 5 down to 1 optimal-sized Parquet file(s).

[3] LIQUID CLUSTERING: Dynamic layout that replaces Partitioning & Z-Order.
-> Evidence: Table 'training_medallion.00_landing.risk_liquid_demo' created with Adaptable Layout.

[4] VACUUM: Safe storage cleanup with 7-day retention.
-> Evidence: Transaction log currently holds 7 versions.
-> Evidence: Vacuum executed successfully. Delta Lake strictly protects data younger than 7 days (168 hours) to guarantee Time Travel safety. Obsolete files older than 7 days will be physically deleted.
